🔹 Ejercicio 1 — Sistema de pedidos con case class

In [1]:
case class Cliente(id: Int, nombre: String, email: String)

case class LineaPedido(producto: String, cantidad: Int, precioUnitario: Double) {
  def subtotal(): Double = cantidad * precioUnitario
}

case class Pedido(numero: Int, cliente: Cliente, lineas: List[LineaPedido]) {
  def total(): Double = lineas.map(_.subtotal()).sum

  def resumen(): String = {
    val cab  = s"=== Pedido #$numero — ${cliente.nombre} ==="
    val body = lineas.map(l =>
      f"  ${l.producto}%-20s ${l.cantidad}%2d ud × ${l.precioUnitario}%7.2f € = ${l.subtotal()}%8.2f €"
    ).mkString("\n")
    val pie  = f"  ${"TOTAL"}%-20s ${""}%2s     ${""}%7s   ${total()}%8.2f €"
    s"$cab\n$body\n${"-" * 55}\n$pie"
  }
}

defined class Cliente
defined class LineaPedido
defined class Pedido

In [3]:
val cliente1 = Cliente(1, "Ana García", "ana@ejemplo.com")

val pedido1 = Pedido(
  numero  = 1001,
  cliente = cliente1,
  lineas  = List(
    LineaPedido("Teclado mecánico",  1, 79.99),
    LineaPedido("Ratón inalámbrico", 2, 29.99),
    LineaPedido("Alfombrilla",       1, 12.50)
  )
)

println(pedido1.resumen())

=== Pedido #1001 — Ana García ===
  Teclado mecánico      1 ud ×   79.99 € =    79.99 €
  Ratón inalámbrico     2 ud ×   29.99 € =    59.98 €
  Alfombrilla           1 ud ×   12.50 € =    12.50 €
-------------------------------------------------------
  TOTAL                                   152.47 €


cliente1: Cliente = Cliente(
  id = 1,
  nombre = "Ana García",
  email = "ana@ejemplo.com"
)
pedido1: Pedido = Pedido(
  numero = 1001,
  cliente = Cliente(id = 1, nombre = "Ana García", email = "ana@ejemplo.com"),
  lineas = List(
    LineaPedido(
      producto = "Teclado mecánico",
      cantidad = 1,
      precioUnitario = 79.99
    ),
    LineaPedido(
      producto = "Ratón inalámbrico",
      cantidad = 2,
      precioUnitario = 29.99
    ),
    LineaPedido(producto = "Alfombrilla", cantidad = 1, precioUnitario = 12.5)
  )
)

In [4]:
// Aplicar descuento del 10% en el precio del teclado con copy
val tecladoRebajado  = pedido1.lineas.head.copy(precioUnitario = 71.99)
val pedido1Rebajado  = pedido1.copy(
  lineas = tecladoRebajado :: pedido1.lineas.tail
)

println(pedido1Rebajado.resumen())
println(s"\nAhorro: ${f"${pedido1.total() - pedido1Rebajado.total()}%.2f"} €")

=== Pedido #1001 — Ana García ===
  Teclado mecánico      1 ud ×   71.99 € =    71.99 €
  Ratón inalámbrico     2 ud ×   29.99 € =    59.98 €
  Alfombrilla           1 ud ×   12.50 € =    12.50 €
-------------------------------------------------------
  TOTAL                                   144.47 €

Ahorro: 8.00 €


tecladoRebajado: LineaPedido = LineaPedido(
  producto = "Teclado mecánico",
  cantidad = 1,
  precioUnitario = 71.99
)
pedido1Rebajado: Pedido = Pedido(
  numero = 1001,
  cliente = Cliente(id = 1, nombre = "Ana García", email = "ana@ejemplo.com"),
  lineas = List(
    LineaPedido(
      producto = "Teclado mecánico",
      cantidad = 1,
      precioUnitario = 71.99
    ),
    LineaPedido(
      producto = "Ratón inalámbrico",
      cantidad = 2,
      precioUnitario = 29.99
    ),
    LineaPedido(producto = "Alfombrilla", cantidad = 1, precioUnitario = 12.5)
  )
)

 Ejercicio 2 — Pattern matching sobre una jerarquía sealed trait

In [5]:
sealed trait EstadoEnvio

case object Preparando                                          extends EstadoEnvio
case class  EnTransito(transportista: String, ubicacion: String) extends EstadoEnvio
case class  Entregado(firma: String)                            extends EstadoEnvio
case class  Incidencia(motivo: String)                          extends EstadoEnvio
case object Devuelto                                            extends EstadoEnvio

defined trait EstadoEnvio
defined object Preparando
defined class EnTransito
defined class Entregado
defined class Incidencia
defined object Devuelto

In [6]:
def descripcionEstado(estado: EstadoEnvio): String = estado match {
  case Preparando              => "📦 El paquete está siendo preparado en el almacén"
  case EnTransito(trans, ubic) => s"🚚 En tránsito con $trans — Última ubicación: $ubic"
  case Entregado(firma)        => s"✅ Entregado y firmado por: $firma"
  case Incidencia(motivo)      => s"⚠️  Incidencia: $motivo"
  case Devuelto                => "↩️  El paquete ha sido devuelto al remitente"
}

defined function descripcionEstado

In [7]:
val historial: List[(String, EstadoEnvio)] = List(
  ("10:00", Preparando),
  ("14:30", EnTransito("MRW", "Madrid - Centro de clasificación")),
  ("18:00", EnTransito("MRW", "Barcelona - Almacén local")),
  ("09:15", Entregado("L. Martínez"))
)

println("=== Seguimiento de envío #ES123456 ===")
for ((hora, estado) <- historial) {
  println(s"  [$hora] ${descripcionEstado(estado)}")
}

=== Seguimiento de envío #ES123456 ===
  [10:00] 📦 El paquete está siendo preparado en el almacén
  [14:30] 🚚 En tránsito con MRW — Última ubicación: Madrid - Centro de clasificación
  [18:00] 🚚 En tránsito con MRW — Última ubicación: Barcelona - Almacén local
  [09:15] ✅ Entregado y firmado por: L. Martínez


historial: List[(String, EstadoEnvio)] = List(
  ("10:00", Preparando),
  (
    "14:30",
    EnTransito(
      transportista = "MRW",
      ubicacion = "Madrid - Centro de clasificación"
    )
  ),
  (
    "18:00",
    EnTransito(transportista = "MRW", ubicacion = "Barcelona - Almacén local")
  ),
  ("09:15", Entregado(firma = "L. Martínez"))
)

🔹 Ejercicio 3 — Calculadora con pattern matching

In [1]:
sealed trait Operacion
case class Sumar(a: Double, b: Double)        extends Operacion
case class Restar(a: Double, b: Double)       extends Operacion
case class Multiplicar(a: Double, b: Double)  extends Operacion
case class Dividir(a: Double, b: Double)      extends Operacion
case class Potencia(base: Double, exp: Int)   extends Operacion

defined trait Operacion
defined class Sumar
defined class Restar
defined class Multiplicar
defined class Dividir
defined class Potencia

In [2]:
def calcular(op: Operacion): String = op match {
  case Sumar(a, b)          => f"$a + $b = ${a + b}%.4f"
  case Restar(a, b)         => f"$a - $b = ${a - b}%.4f"
  case Multiplicar(a, b)    => f"$a × $b = ${a * b}%.4f"
  case Dividir(_, 0)        => "❌ Error: división por cero"
  case Dividir(a, b)        => f"$a ÷ $b = ${a / b}%.4f"
  case Potencia(base, exp)  =>
    val resultado = math.pow(base, exp)
    f"$base ^ $exp = $resultado%.4f"
}

val operaciones = List(
  Sumar(15.0, 7.5),
  Restar(100.0, 43.25),
  Multiplicar(4.0, 3.5),
  Dividir(22.0, 7.0),
  Dividir(5.0, 0.0),
  Potencia(2.0, 10)
)

println("=== Calculadora Scala ===")
for (op <- operaciones) println(s"  ${calcular(op)}")

=== Calculadora Scala ===
  15.0 + 7.5 = 22.5000
  100.0 - 43.25 = 56.7500
  4.0 × 3.5 = 14.0000
  22.0 ÷ 7.0 = 3.1429
  ❌ Error: división por cero
  2.0 ^ 10 = 1024.0000


defined function calcular
operaciones: List[Product with Operacion with Serializable] = List(
  Sumar(a = 15.0, b = 7.5),
  Restar(a = 100.0, b = 43.25),
  Multiplicar(a = 4.0, b = 3.5),
  Dividir(a = 22.0, b = 7.0),
  Dividir(a = 5.0, b = 0.0),
  Potencia(base = 2.0, exp = 10)
)

🔹 Ejercicio 4 — Sistema de notificaciones

In [3]:
case class Usuario(id: Int, nombre: String, email: String)

sealed trait Notificacion
case class Email(destinatario: Usuario,
                 asunto: String,
                 cuerpo: String)         extends Notificacion
case class SMS(telefono: String,
               mensaje: String)          extends Notificacion
case class Push(dispositivo: String,
                titulo: String,
                texto: String)           extends Notificacion
case object SinNotificacion              extends Notificacion

defined class Usuario
defined trait Notificacion
defined class Email
defined class SMS
defined class Push
defined object SinNotificacion

In [10]:
def enviarNotificacion(n: Notificacion): String = n match {
  case Email(user, asunto, cuerpo) =>
    s"📧 EMAIL → ${user.email}\n   Asunto: $asunto\n   Cuerpo: ${cuerpo.take(40)}..."
  case SMS(tel, msg) =>
    s"📱 SMS → $tel\n   Mensaje: ${msg.take(50)}"
  case Push(disp, titulo, texto) =>
    s"🔔 PUSH → $disp\n   $titulo: $texto"
  case SinNotificacion =>
    "🔕 Sin notificación configurada"
}

def prioridad(n: Notificacion): Int = n match {
  case Email(_, _, _)  => 3
  case SMS(_, _)       => 2
  case Push(_, _, _)   => 1
  case SinNotificacion => 0
}

defined function enviarNotificacion
defined function prioridad

In [ ]:
val usuario1 = Usuario(1, "Ana López",   "ana@ejemplo.com")
val usuario2 = Usuario(2, "Carlos Ruiz", "carlos@ejemplo.com")

val notificaciones: List[Notificacion] = List(
  Email(usuario1, "Confirmación de pedido",
        "Tu pedido #1001 ha sido confirmado y está siendo preparado."),
  SMS("+34 600 123 456",
      "Tu paquete llegará mañana entre las 9h y las 14h."),
  Push("iPhone-Carlos", "Nueva oferta",
       "30% de descuento en tecnología este fin de semana"),
  SinNotificacion
)

println("=== Centro de notificaciones ===\n")
for (n <- notificaciones.sortBy(-prioridad(_))) {
  println(enviarNotificacion(n))
  println()
}

=== Centro de notificaciones ===

📧 EMAIL → ana@ejemplo.com
   Asunto: Confirmación de pedido
   Cuerpo: Tu pedido #1001 ha sido confirmado y est...

📱 SMS → +34 600 123 456
   Mensaje: Tu paquete llegará mañana entre las 9h y las 14h.

🔔 PUSH → iPhone-Carlos
   Nueva oferta: 30% de descuento en tecnología este fin de semana

🔕 Sin notificación configurada



usuario1: Usuario = Usuario(
  id = 1,
  nombre = "Ana López",
  email = "ana@ejemplo.com"
)
usuario2: Usuario = Usuario(
  id = 2,
  nombre = "Carlos Ruiz",
  email = "carlos@ejemplo.com"
)
notificaciones: List[Notificacion] = List(
  Email(
    destinatario = Usuario(
      id = 1,
      nombre = "Ana López",
      email = "ana@ejemplo.com"
    ),
    asunto = "Confirmación de pedido",
    cuerpo = "Tu pedido #1001 ha sido confirmado y está siendo preparado."
  ),
  SMS(
    telefono = "+34 600 123 456",
    mensaje = "Tu paquete llegará mañana entre las 9h y las 14h."
  ),
  Push(
    dispositivo = "iPhone-Carlos",
    titulo = "Nueva oferta",
    texto = "30% de descuento en tecnología este fin de semana"
  ),
  SinNotificacion
)

: 

Ejercicio 1 propuesto

Sistema de calificación de alumno y calculo de promedio final.

In [1]:
case class alumno(id: Int, nombre: String, email: String)

case class calificacion(curso: String, nota: Double, creditos: Int){
    def subtotal(): Double = nota * creditos
 }

case class expediente(numero: Int, alumno: alumno, calificaciones: List[calificacion]){
    def promedio(): Double = calificaciones.map(_.subtotal()).sum / calificaciones.map(_.creditos).sum

    def resumen(): String = {
        val cab  = s"=== Expediente #$numero — ${alumno.nombre} ==="
        val body = calificaciones.map(c =>
            f"  ${c.curso}%-20s ${c.nota}%2.2f   ${c.creditos}%2d "
        ).mkString("\n")
        val pie  = f"  ${"PROMEDIO"}%-20s ${""}%2s   ${promedio()}%8.2f"
        s"$cab\n$body\n${"-" * 55}\n$pie"
    }
 }

defined class alumno
defined class calificacion
defined class expediente

In [ ]:
val alumno1 = alumno(1, "Ana García", "agarcia001@ej.es")

val expediente1 = expediente(
  numero = 2024001,
  alumno = alumno1,
  calificaciones = List(
    calificacion("Matemáticas", 8.5, 6),
    calificacion("Física",       7.0, 4),
    calificacion("Literatura",   9.0, 5)
  )
)

println(expediente1.resumen())

=== Expediente #2024001 — Ana García ===
  Matemáticas          8.50    6 
  Física               7.00    4 
  Literatura           9.00    5 
-------------------------------------------------------
  PROMEDIO                      8.27


alumno1: alumno = alumno(
  id = 1,
  nombre = "Ana García",
  email = "agarcia001@ej.es"
)
expediente1: expediente = expediente(
  numero = 2024001,
  alumno = alumno(id = 1, nombre = "Ana García", email = "agarcia001@ej.es"),
  calificaciones = List(
    calificacion(curso = "Matemáticas", nota = 8.5, creditos = 6),
    calificacion(curso = "Física", nota = 7.0, creditos = 4),
    calificacion(curso = "Literatura", nota = 9.0, creditos = 5)
  )
)

: 

Ejercicio 2 - Pattern matching sobre una jerarquia sealed trait


In [1]:
sealed trait promocion

case class descuentoDirecto(precio: Double)                extends promocion
case class cuponRegalo(nombre: String, precio: Double)     extends promocion
case object sinPromocion                                    extends promocion

defined trait promocion
defined class descuentoDirecto
defined class cuponRegalo
defined object sinPromocion

In [2]:
def aplicarPromocion(promo: promocion): String = promo match {
  case descuentoDirecto(p)        => "El cliente tiene un descuento directo del 10%: " + (p * 0.9)
  case cuponRegalo(n, p)          => s"$n tiene un cupón de regalo y pagará: " + (p - 5.0)
  case sinPromocion               => s"El cliente no tiene promoción" 
}

defined function aplicarPromocion

In [ ]:
val clientes = List(
  descuentoDirecto(100.0),
  cuponRegalo("Anita", 15.0),
  sinPromocion
)

println("=== Promociones disponibles ===\n")
for (promo <- clientes) {   
     println(aplicarPromocion(promo))
    }        

=== Promociones disponibles ===

El cliente tiene un descuento directo del 10%: 90.0
Anita tiene un cupón de regalo y pagará: 10.0
El cliente no tiene promoción


clientes: List[Product with promocion with Serializable] = List(
  descuentoDirecto(precio = 100.0),
  cuponRegalo(nombre = "Anita", precio = 15.0),
  sinPromocion
)

: 

Ejercicio 03 - con pattern matching

Gestor de usuarios

In [1]:
sealed trait tipoUsuario
case object administrador extends tipoUsuario
case object editor        extends tipoUsuario
case object visitante     extends tipoUsuario

defined trait tipoUsuario
defined object administrador
defined object editor
defined object visitante

In [2]:
sealed trait recurso
case class documento(nombre: String, contenido: String) extends recurso
case class configuracion(nivelSeguridad: Int)           extends recurso
case object papelera                                    extends recurso



defined trait recurso
defined class documento
defined class configuracion
defined object papelera

In [1]:
//Seleaded traite del tipo de usuario
sealed trait TipoUsuario
case object Administrador extends TipoUsuario
case object Editor        extends TipoUsuario
case object Visitante     extends TipoUsuario

//seleated trait de los recursos que puede acceder segun el tipo de usuario
sealed trait Recurso
case class Documento(nombre: String, contenido: String) extends Recurso
case class Configuracion(nivelSeguridad: Int)           extends Recurso
case object Papelera                                    extends Recurso

//definicion de la funcion que valida el acceso segun el tipo de usuario y el recurso al que quiere acceder
def validarAcceso(usu: TipoUsuario, rec: Recurso): String = (usu, rec) match {
  case (Administrador, _)                                                 => "Acceso concedido: el administrador tiene acceso a todos los recursos."
  case (Editor, Documento(_, _))                                          => "Acceso concedido: el editor puede acceder a documentos."
  case (Editor, Configuracion(nivel)) if nivel <= 2                       => "Acceso concedido: el editor puede acceder a configuraciones de nivel 2 o inferior."
  case (Visitante, Documento(nombre, _)) if nombre.startsWith("publico")  => "Acceso concedido: el visitante puede acceder a documentos públicos."
  case _                                                                  => "Acceso denegado: no tienes permiso para acceder a este recurso."
}

defined trait TipoUsuario
defined object Administrador
defined object Editor
defined object Visitante
defined trait Recurso
defined class Documento
defined class Configuracion
defined object Papelera
defined function validarAcceso

In [2]:
val ListaNuevosUsuarios = List(
  (Administrador, Documento("confidencial.txt", "Contenido secreto")),
  (Editor, Documento("publico.txt", "Contenido público")),
  (Visitante, Documento("publico_informacion.txt", "Información para visitantes")),
  (Visitante, Configuracion(1)),
  (Editor, Configuracion(3)),
  (Editor, Configuracion(2)),
  (Visitante, Configuracion(1)),
  (Administrador, Papelera),
  (Editor, Papelera),
  (Visitante, Papelera)
)

println("=== Validación de acceso a recursos ===\n")
for ((usu, rec) <- ListaNuevosUsuarios) {
  println(s"Usuario: $usu, Recurso: $rec")
  println(s"Resultado: ${validarAcceso(usu, rec)}\n")
}

=== Validación de acceso a recursos ===

Usuario: Administrador, Recurso: Documento(confidencial.txt,Contenido secreto)
Resultado: Acceso concedido: el administrador tiene acceso a todos los recursos.

Usuario: Editor, Recurso: Documento(publico.txt,Contenido público)
Resultado: Acceso concedido: el editor puede acceder a documentos.

Usuario: Visitante, Recurso: Documento(publico_informacion.txt,Información para visitantes)
Resultado: Acceso concedido: el visitante puede acceder a documentos públicos.

Usuario: Visitante, Recurso: Configuracion(1)
Resultado: Acceso denegado: no tienes permiso para acceder a este recurso.

Usuario: Editor, Recurso: Configuracion(3)
Resultado: Acceso denegado: no tienes permiso para acceder a este recurso.

Usuario: Editor, Recurso: Configuracion(2)
Resultado: Acceso concedido: el editor puede acceder a configuraciones de nivel 2 o inferior.

Usuario: Visitante, Recurso: Configuracion(1)
Resultado: Acceso denegado: no tienes permiso para acceder a este 

ListaNuevosUsuarios: List[(Product with TipoUsuario with Serializable, Product with Recurso with Serializable)] = List(
  (
    Administrador,
    Documento(nombre = "confidencial.txt", contenido = "Contenido secreto")
  ),
  (Editor, Documento(nombre = "publico.txt", contenido = "Contenido público")),
  (
    Visitante,
    Documento(
      nombre = "publico_informacion.txt",
      contenido = "Información para visitantes"
    )
  ),
  (Visitante, Configuracion(nivelSeguridad = 1)),
  (Editor, Configuracion(nivelSeguridad = 3)),
  (Editor, Configuracion(nivelSeguridad = 2)),
  (Visitante, Configuracion(nivelSeguridad = 1)),
  (Administrador, Papelera),
  (Editor, Papelera),
  (Visitante, Papelera)
)